# 00 · Setup — WA+ West Rapti training  (run once, ~5 minutes)

This notebook installs everything into **My Drive › west_rapti** and checks that it works.

1. **File › Save a copy in Drive** (so you can edit it).
2. In cell ① type your **Earth Engine project ID** and your name.
3. **Runtime › Run all.** Approve the Google sign-in pop-ups (Drive, then Earth Engine).
4. Every check must end with ✅. Then open **My Drive › west_rapti › Lab1_delineate_interactive.ipynb**.

Re-running this notebook is safe: it refreshes the shared code but **never overwrites your own notebooks or outputs**.

In [ ]:
#@title ① Your details  (edit, then Runtime ▸ Run all)
EE_PROJECT       = 'your-ee-project-id'   #@param {type:"string"}
PARTICIPANT_NAME = ''                     #@param {type:"string"}

# --- set once by the trainer before sharing this notebook (participants: leave as is) ---
PACKAGE_URL  = ''    # Google Drive link ("Anyone with the link") to WA_Plus_Participant_Package.zip
HYDROMET_URL = ''    # optional: link to the DHM HydroMetData zip (only if sharing is permitted)

In [ ]:
#@title ② Connect Google Drive and check your details
import os, sys, json, glob, shutil, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')
ROOT   = '/content/drive/MyDrive'
KW_DIR = os.path.join(ROOT, 'west_rapti')
os.makedirs(KW_DIR, exist_ok=True)
EE_PROJECT = EE_PROJECT.strip()
if not EE_PROJECT or EE_PROJECT == 'your-ee-project-id':
    raise SystemExit('❌ Type your Earth Engine project ID in cell ① (see README_participants.md, step 0), then Run all again.')
try:
    free_gb = shutil.disk_usage(ROOT).free / 1e9
    print(f'Drive free space ≈ {free_gb:.1f} GB' + ('   ⚠️ the labs need ~5 GB — free some space' if free_gb < 5 else ''))
except Exception:
    pass
print('✅ Drive connected:', KW_DIR)

In [ ]:
#@title ③ Install / update the training package
ENGINE = {'west_rapti.py', 'westrapti_prep.py', 'selftest_west_rapti.py', 'README_participants.md', '00_Setup.ipynb'}

def install_package(zpath, root):
    """Install the package into My Drive/west_rapti, whatever the folder layout inside the zip.
    Shared code is always refreshed; a notebook the participant already has is never overwritten."""
    if not zipfile.is_zipfile(zpath):
        head = open(zpath, 'rb').read(400).lower()
        why = 'a Google sign-in / permission page' if b'<html' in head else 'not a zip file'
        raise SystemExit(f'❌ The package download is {why}. Ask the trainer to set the zip link to '
                         '"Anyone with the link · Viewer", then Run all again.')
    target = os.path.join(root, 'west_rapti'); os.makedirs(target, exist_ok=True)
    new, kept = [], []
    with zipfile.ZipFile(zpath) as z:
        names = [m for m in z.namelist() if not m.endswith('/') and '__MACOSX' not in m and '/._' not in m]
        eng = sorted((m for m in names if os.path.basename(m) == 'west_rapti.py'), key=len)
        if not eng:
            tops = sorted({m.split('/')[0] for m in names})[:6]
            raise SystemExit(f'❌ This zip has no west_rapti.py — wrong file? It contains: {tops}')
        prefix = eng[0][:-len('west_rapti.py')]            # 'west_rapti/', '' or 'WA_Plus_Nepal_Training/code/'
        for m in names:
            if m.startswith(prefix) and '__pycache__' not in m:
                rel = m[len(prefix):]
            elif m.endswith('.ipynb'):
                rel = os.path.basename(m)                   # notebooks kept elsewhere in the zip (trainer kit)
            else:
                continue
            dst, name = os.path.join(target, rel), os.path.basename(rel)
            refresh = name in ENGINE or rel.startswith('slides/')
            if os.path.exists(dst) and not refresh:
                kept.append(name); continue
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with z.open(m) as src_, open(dst, 'wb') as d:
                shutil.copyfileobj(src_, d)
            new.append(name)
    if not new and not kept:
        raise SystemExit('❌ Nothing was installed from the zip — send the trainer a screenshot of this cell.')
    return new, kept

zpath = None
if PACKAGE_URL:
    import gdown
    zpath = gdown.download(PACKAGE_URL, '/content/package.zip', quiet=True, fuzzy=True)
    if not zpath:
        raise SystemExit('❌ Could not download the package — ask the trainer to check the link sharing ("Anyone with the link").')
else:
    found = sorted(glob.glob(os.path.join(ROOT, 'WA_Plus_Participant_Package*.zip')))
    zpath = found[-1] if found else None
if zpath:
    new, kept = install_package(zpath, ROOT)
    print(f'✅ installed/updated {len(new)} files' + (f' | kept your own copies of {len(kept)} files' if kept else ''))
    print('   in My Drive › west_rapti:', sorted(f for f in os.listdir(KW_DIR) if f.endswith(('.ipynb', '.py'))))
elif not os.path.exists(os.path.join(KW_DIR, 'west_rapti.py')):
    raise SystemExit('❌ Package not found: upload WA_Plus_Participant_Package.zip to the top level of My Drive, then Run all again.')
else:
    print('✅ package already installed (no zip found to update from)')

for stale in ('/content/west_rapti.py', '/content/westrapti_prep.py'):      # old copies that would shadow Drive
    if os.path.exists(stale): os.remove(stale)
json.dump({'ee_project': EE_PROJECT, 'name': PARTICIPANT_NAME.strip()},
          open(os.path.join(KW_DIR, 'participant.json'), 'w'), indent=2)
print('✅ participant.json saved — every lab reads your Earth Engine project from it')

In [ ]:
#@title ④ Install Python libraries  (~2 minutes)
!pip -q install pysheds rioxarray geemap geedim scikit-image rasterio wapordl dem-stitcher earthengine-api gdown
print('✅ libraries installed')

In [ ]:
#@title ⑤ Self-test the WA+ engine  (no data needed, ~10 s)
r = subprocess.run([sys.executable, 'selftest_west_rapti.py'], cwd=KW_DIR, capture_output=True, text=True)
print(r.stdout[-2000:])
if 'ALL PASSED' not in r.stdout:
    print(r.stderr[-2000:])
    raise SystemExit('❌ Engine self-test failed — send a screenshot of this cell to the trainer.')
print('✅ engine OK')

In [ ]:
#@title ⑥ Check Earth Engine  (approve the sign-in pop-up)
import ee
try:
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)
    n = ee.ImageCollection('COPERNICUS/S1_GRD').filterDate('2024-01-01', '2024-01-02').size().getInfo()
    print(f'✅ Earth Engine works with project "{EE_PROJECT}" (test query: {n} Sentinel-1 scenes)')
except Exception as e:
    print('❌ Earth Engine is not ready:', str(e)[:400])
    print('   Most common fixes: (1) register the Cloud project for Earth Engine at https://code.earthengine.google.com/register'
          ' (non-commercial use); (2) check the project ID spelling in cell ①; (3) sign in with the same Google account.')
    raise SystemExit

In [ ]:
#@title ⑦ DHM gauge data (rainfall + discharge)
def find_hydromet(base):
    hits = glob.glob(os.path.join(base, '**', 'rapti_rain.xlsx'), recursive=True)
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None      # .../HydroMetData/Temp_PPT/rapti_rain.xlsx

hm = find_hydromet(KW_DIR)
if hm is None and HYDROMET_URL:
    import gdown
    hz = gdown.download(HYDROMET_URL, '/content/hydromet.zip', quiet=True, fuzzy=True)
    if hz:
        zipfile.ZipFile(hz).extractall(KW_DIR); hm = find_hydromet(KW_DIR)
if hm:
    r = subprocess.run([sys.executable, 'westrapti_prep.py', '--src', hm, '--out', KW_DIR,
                        '--start', '2021-01-01', '--end', '2025-01-01'], cwd=KW_DIR, capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode == 0 and os.path.exists(os.path.join(KW_DIR, 'discharge_outlet.csv')):
        print('✅ gauge files ready: dhm_stations.csv, dhm_precip.csv, discharge_monthly.csv, discharge_outlet.csv')
    else:
        print(r.stderr[-1500:]); print('⚠️ gauge preparation failed — tell the trainer; Labs 1 and 3 still work.')
else:
    print('⚠️ HydroMetData not found in My Drive › west_rapti. Labs 1 and 3 work without it; Labs 2, 4 and 6 need it '
          'for gauge bias-correction and calibration — ask the trainer for the data.')

In [ ]:
#@title ⑧ Summary
labs = sorted(glob.glob(os.path.join(KW_DIR, 'Lab*.ipynb')))
print('Notebooks in My Drive › west_rapti:'); [print('   ', os.path.basename(p)) for p in labs]
print('\n✅ Setup complete. Open Lab1_delineate_interactive.ipynb (double-click → Open with Google Colaboratory),')
print('   then Runtime ▸ Run all. Run the labs in order 1 → 6; each saves to west_rapti/outputs/.')